In [1]:
# Step 1: Import libraries and load the API key
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()   # reads the .env file
api_key = os.getenv("TMDB_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [2]:
# Step 2: Test the API with one movie (Avengers: Endgame)
url = "https://api.themoviedb.org/3/movie/299534"
response = requests.get(url, params={"api_key": api_key})

print("Status code:", response.status_code)
print("Title:", response.json().get("title"))

Status code: 200
Title: Avengers: Endgame


In [3]:
# Step 3: Fetch all movies from the lab list
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
             420818, 24428, 168259, 99861, 284054, 12445,
             181808, 330457, 351286, 109445, 321612, 260513]

movies = []   # empty list to collect each movie

for movie_id in movie_ids:
    url = "https://api.themoviedb.org/3/movie/" + str(movie_id)
    params = {"api_key": api_key, "append_to_response": "credits"}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        movies.append(response.json())
        print("OK:", movie_id)
    else:
        print("Not found:", movie_id)

df = pd.DataFrame(movies)
print("Number of movies:", len(df))

Not found: 0
OK: 299534
OK: 19995
OK: 140607
OK: 299536
OK: 597
OK: 135397
OK: 420818
OK: 24428
OK: 168259
OK: 99861
OK: 284054
OK: 12445
OK: 181808
OK: 330457
OK: 351286
OK: 109445
OK: 321612
OK: 260513
Number of movies: 18


In [4]:
# Step 4: Drop irrelevant columns
columns_to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
df = df.drop(columns=columns_to_drop)

print("Columns left:", len(df.columns))
print(df.columns.tolist())

Columns left: 23
['backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'id', 'origin_country', 'original_language', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'credits']


In [5]:
# Step 5: Extract names from JSON-like columns

def get_names(value, key="name"):
    # value is a list of dictionaries -> join the names with "|"
    if type(value) != list:
        return None
    names = []
    for item in value:
        names.append(item[key])
    return "|".join(names)

def get_collection(value):
    # value is one dictionary, or empty if the movie has no collection
    if type(value) == dict:
        return value["name"]
    return None

df["genres"] = df["genres"].apply(get_names)
df["production_countries"] = df["production_countries"].apply(get_names)
df["production_companies"] = df["production_companies"].apply(get_names)
df["spoken_languages"] = df["spoken_languages"].apply(get_names, key="english_name")
df["belongs_to_collection"] = df["belongs_to_collection"].apply(get_collection)

df[["title", "genres", "belongs_to_collection", "spoken_languages"]].head()

,title,genres,belongs_to_collection,spoken_languages
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,English|Japanese|Xhosa
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,English|Spanish
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,English
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,English|Xhosa
4,Titanic,Drama|Romance,None,English|French|German|Swedish|Italian|Russian


In [6]:
# Step 6: Extract cast and director from credits

def get_cast(credits):
    # names of all actors, joined with "|"
    names = []
    for person in credits["cast"]:
        names.append(person["name"])
    return "|".join(names)

def get_director(credits):
    # find the crew member whose job is "Director"
    for person in credits["crew"]:
        if person["job"] == "Director":
            return person["name"]
    return None

df["cast"] = df["credits"].apply(get_cast)
df["cast_size"] = df["credits"].apply(lambda c: len(c["cast"]))
df["director"] = df["credits"].apply(get_director)
df["crew_size"] = df["credits"].apply(lambda c: len(c["crew"]))

df = df.drop(columns=["credits"])

df[["title", "director", "cast_size", "crew_size"]].head()

,title,director,cast_size,crew_size
0,Avengers: Endgame,Anthony Russo,103,613
1,Avatar,James Cameron,68,999
2,Star Wars: The Force Awakens,J.J. Abrams,99,270
3,Avengers: Infinity War,Joe Russo,69,737
4,Titanic,James Cameron,116,271
